In [99]:
# Carregamento das bibliotecas
import pandas as pd
import numpy as np
import urllib.parse
import xgboost as xgb
import shap 
import matplotlib.pyplot as plt
import warnings 

from sqlalchemy import create_engine
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [100]:
def get_db_connection():
    """
    Cria e retorna a engine de conexão com o PostgreSQL
    Segurança: Em um ambiente de produção, NUNCA deixe as credencias hardcoded
    Sempre utilize variáveis de ambiente (ex: os.environ.get('DB_PASS')).
    """

    # Placeholder para as credenciais do banco local
    db_user = "postgres"
    # a função quote_plus para proteger sua senha de caracteres especiais
    db_pass = urllib.parse.quote_plus("84570815eE@")
    db_host = '127.0.0.1'
    db_port = '5432'
    db_name = 'inventory_intelligence'

    # Adicionando o host explicitamente no final para desativar a tentativa de Unix Socket
    connection_url = f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}?host={db_host}"

    engine = create_engine(connection_url, 
                           connect_args={'client_encoding': 'utf8'}) # Força o cliente a falar UTF-8
    return engine

In [101]:
def load_data(engine):
    """
    Executa a query e carrega os dados em um DataFrame
    """
    query = """
        WITH base_diaria AS (
            SELECT
                c.data_atual,
                p.produto_id,
                p.nome,
                p.estoque_inicial,
                COALESCE(SUM(v.quantidade), 0) AS qtd_vendida_dia
            FROM calendario c
            CROSS JOIN produtos p
            LEFT JOIN vendas v
                ON c.data_atual = v.data_venda
                AND p.produto_id = v.produto_id
            GROUP BY c.data_atual, p.produto_id, p.nome, p.estoque_inicial
        )
        SELECT
            data_atual,
            produto_id,
            nome,
            qtd_vendida_dia,
            ROUND(AVG(qtd_vendida_dia) OVER(PARTITION BY produto_id ORDER BY data_atual ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS media_movel_7d,
            COALESCE(LAG(qtd_vendida_dia, 7) OVER(PARTITION BY produto_id ORDER BY data_atual), 0) AS lag_vendas_7d,
            (estoque_inicial - SUM(qtd_vendida_dia) OVER(PARTITION BY produto_id ORDER BY data_atual ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS estoque_restante_estimado
        FROM base_diaria
        ORDER BY produto_id, data_atual;  
        """
    return pd.read_sql(query, engine)


In [102]:
def build_time_features(df):
    """
    Extrai os componentes de data e cria representações cíclicas para o modelo
    """
    # 1. Conversão garantida para o datetime
    df['data_atual'] = pd.to_datetime(df['data_atual'], errors='coerce')

    # 2. Componentes básicos
    # dt.dayofweek retorna 0 para Segunda-feira e 6 para Domingo
    df['dia_da_semana'] = df['data_atual'].dt.dayofweek
    df['mes'] = df['data_atual'].dt.month
    df['fim_de_semana'] = df['dia_da_semana'].isin([5, 6])

    # 3. Engenharia de Features
    """
    Por quê fazer isso? Para algoritmos de ML, o dia 365 (31/dez) e o dia (1/jan) parecem numericamente
    muitos distantes (365 - 1 = 364 de distância). Na vida real, é apenas 1 dia de diferença,
    mas ao projetarmos o "dia do ano" num círculo trigonométrico usando seno e cosseno,
    as datas do fim do ano se conectam suavemente com as do início do ano seguinte,
    mantendo a correta noção espacial da sazonalidade
    """

    dias_no_ano = 365.25 #.25 para acomodar anos bissextos
    dia_do_ano = df['data_atual'].dt.dayofyear

    # Convertendo a escala de dias (1-365) para o círculo radiano (0 a 2*pi)
    df['sin_dia_ano'] = np.sin(2 * np.pi * dia_do_ano / dias_no_ano)
    df['cos_dia_ano'] = np.cos(2 * np.pi * dia_do_ano / dias_no_ano)

    return df


In [103]:
def handle_outliers(df, column_name='qtd_vendida_dia', group_by_col='produto_id', threshold=3):
    """
    Identifica e substitui anomalias extremas (acima de 'threshold' desvios padrão)
    pela mediana do próprio produto, evitando impacto negativo no modelo preditivo.
    """
    # Criamos uma cópia para evitar warnings de SettingWithCopy no Pandas
    df_clean = df.copy()

    # Tratamento de Outliers (Z-score > 3 substituídos pela mediana do produto)
    mean = df.groupby('produto_id', observed=False)['qtd_vendida_dia'].transform('mean')
    std = df.groupby('produto_id', observed=False)['qtd_vendida_dia'].transform('std')
    median = df.groupby('produto_id', observed=False)['qtd_vendida_dia'].transform('median')

    # Cálculo do Z-Score para identificar quão longe cada ponto está da média de seu produto
    z_score = np.abs((df['qtd_vendida_dia'] - mean) / std)

    # Filtro lógico: Onde o Z-Score passar do threshold, consideramos como "Pico Anômalo"
    is_outlier = z_score > 3

    # Substituímos os outliers pela mediana (pois a mediana é uma métrica resistente a valores extremos)
    df.loc[is_outlier, 'qtd_vendida_dia'] = median[is_outlier]

    return df_clean


In [104]:
# Ponto de entrada padrão em scripts python
if __name__ == "__main__":
    def main():
        # Pipeline principal
        engine = get_db_connection()

        print("Iniciando ingestão de dados...")
        df = load_data(engine)

        print("Criando features temporais e cíclicas...")
        df_features = build_time_features(df)

        print("Tratando outliers...")
        df_model = handle_outliers(df_features)

        print("Exportanto dataset de modelagem...")
        df_model.to_csv('base_modelagem.csv', index=False, encoding='utf-8')

        print("Pipeline concluido com sucesso")
        return df_model

    # Ponto de entrada padrão em scripts python
    if __name__ == "__main__":
        df_model = main()

print(df_model.head())

Iniciando ingestão de dados...
Criando features temporais e cíclicas...
Tratando outliers...
Exportanto dataset de modelagem...
Pipeline concluido com sucesso
  data_atual  produto_id              nome  qtd_vendida_dia  media_movel_7d  \
0 2026-04-10           1  Notebook Ultra X               10            10.0   
1 2026-04-11           1  Notebook Ultra X               19            14.5   
2 2026-04-12           1  Notebook Ultra X               43            24.0   
3 2026-04-13           1  Notebook Ultra X               10            20.5   
4 2026-04-14           1  Notebook Ultra X               10            18.4   

   lag_vendas_7d  estoque_restante_estimado  dia_da_semana  mes  \
0              0                      790.0              4    4   
1              0                      771.0              5    4   
2              0                      728.0              6    4   
3              0                      718.0              0    4   
4              0               

In [105]:
# Boas práticas: ignorar avisos não críticos para manter o console limpo
warnings.filterwarnings('ignore')

def load_and_prepare_data(filepath='../data/base_modelagem.csv'):
    """
    Carrega os dados e define o target e as features preditivas
    """
    print("1. Carregando dados...")
    df = pd.read_csv(filepath)
    df['data_atual'] = pd.to_datetime(df['data_atual'], errors='coerce')
    
    # Ordenação estrita por data para garantir a cronologia correta
    df = df.sort_values(by=['produto_id', 'data_atual']).reset_index(drop=True)

    # Shift no estoque para evitar Data Leakage
    # O modelo precisa ver o "estoque no início do dia", não o que sobrou no final
    df['estoque_inicio_dia'] = df.groupby('produto_id')['estoque_restante_estimado'].shift(1)

    # Operações vetorizadas no lugar do .apply() evitam a perda de coluna/índice
    df['num_dia'] = df.groupby('produto_id').cumcount()
    df = df[df['num_dia'] >= 7].reset_index(drop=True)

    # dropando a coluna de estoque original que causaria o vazamento
    df = df.drop(columns=['num_dia', 'estoque_restante_estimado'])
        
    return df

In [106]:
def temporal_split(df, target_col='qtd_vendida_dia', drop_cols=['nome', 'data_atual']):
    """
    Realiza o corte temporal dos dados

    Porque usar a divisão temporal e não aleatória no train_test_split?
    Em séries temporais e problemas de estoque, a ordem dos eventos é vital
    se usarmos divisão aleatória, dados de 'amanhã' podem acabar no conjunto de treino,
    ensinando ao modelo o futuro antes dele tentar prever o hoje. Isso se chama
    'Look-ahead bias' (vazamentos de dados futuros). O modelo ficaria perfeito no
    teste, mas falharia em produção 
    """

    print("2. Realizando a validação temporal (Evitando Look-ahead bias)...")

    # Define o ponto de corte como os últimos 7 dias da base
    data_max = df['data_atual'].max()
    split_date = data_max - pd.Timedelta(days=7)

    # Divisão
    train_df = df[df['data_atual'] < split_date]
    test_df = df[df['data_atual'] >= split_date]

    # Remoção das colunas não preditivas (incluindo a 'nome')
    cols_to_drop = ['data_atual', 'qtd_vendida_dia', 'nome']
    features = [col for col in df.columns if col not in cols_to_drop]

    X_train = train_df[features]
    y_train = train_df[target_col]
    X_test = test_df[features]
    y_test = test_df[target_col]

    return train_df, test_df, X_train, y_train, X_test, y_test, features

In [107]:
def train_and_evaluate(X_train, y_train, X_test, y_test):
    """
    Treina o algoritmo XGBoost e avalia o desempenho
    """
    print("3. Treinando o modelo XGBoost...")
    # XGBoost lida muito bem com: não-linearidade e relações complexas
    model = xgb.XGBRegressor(n_estimators=200,
                             learning_rate=0.1,
                             max_depth=5,
                             random_state=42,
                             tree_method='hist', # Exigido para variáveis categóricas nativas
                             enable_categorical=True, # Permite que XGBoost lide com 'category' dtype
                             objective='reg:squarederror')
    
    model.fit(X_train, y_train)

    # Avaliação
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f" MAE (Erro médio absoluto): {mae:.2f}")
    print(f" RMSE (Raiz do Erro Quadrático médio): {rmse:.2f}")

    return model, y_pred

In [108]:
def explain_with_shap(model, X_test):
    """
    Calcula os SHAP values para IA explicavel
    """
    print("5. Calculando SHAP values...")
    # Utilizando TreeExplainer (Otimizado para XGBoost/Árvores)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    # Gera e salva o gráfico de impacto geral das features
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, show=False)
    plt.savefig('../img/shap_summary_plot.png', bbox_inches='tight')
    plt.close()

    return shap_values

In [109]:
def export_for_powerbi(test_df, X_test, y_test, y_pred, shap_values):
    """
    Cria a tabela final extraindo dinamicamente as top 3 features
    mais importantes de cada previsão para tomada de decisão no PowerBI
    """ 
    print("6. Preparando dados finais para o PowerBI...")

    df_export = pd.DataFrame({'data_atual': test_df['data_atual'].dt.strftime('%Y-%m-%d').values,
                              'produto_id': test_df['produto_id'].values,
                              'valor_real': y_test.values,
                              'previsao_modelo': np.round(y_pred, 2)})
    
    # shap_values oriundos do TreeExplainer já são do tipo numpy array
    shap_matrix = shap_values
    feature_name = np.array(X_test.columns)

    # Ordena indices do maior impacto absoluto para o menor
    top3_indices = np.argsort(-np.abs(shap_matrix), axis=1)[:, :3]

    f1_names, f1_vals = [], []
    f2_names, f2_vals = [], []
    f3_names, f3_vals = [], []

    for i in range(len(test_df)):
        idx1, idx2, idx3 = top3_indices[i]

        f1_names.append(feature_name[idx1])
        f1_vals.append(round(shap_matrix[i, idx1], 4))

        
        f2_names.append(feature_name[idx2])
        f2_vals.append(round(shap_matrix[i, idx2], 4))

        
        f3_names.append(feature_name[idx3])
        f3_vals.append(round(shap_matrix[i, idx3], 4))

    df_export['top1_feature'] = f1_names
    df_export['top1_impacto_shap'] = f1_vals
    df_export['top2_feature'] = f2_names
    df_export['top2_impacto_shap'] = f2_vals
    df_export['top3_feature'] = f3_names
    df_export['top3_impacto_shap'] = f3_vals

    df_export.to_csv('predicoes_inventory.csv', index=False)


def main():
    df = load_and_prepare_data()
    train_df, test_df, X_train, y_train, X_test, y_test, features = temporal_split(df)
    model, y_pred = train_and_evaluate(X_train, y_train, X_test, y_test)
    shap_vals = explain_with_shap(model, X_test)
    export_for_powerbi(test_df, X_test, y_test, y_pred, shap_vals)

if __name__ == "__main__":
    main()

print("7. O processo foi concluido")

1. Carregando dados...
2. Realizando a validação temporal (Evitando Look-ahead bias)...
3. Treinando o modelo XGBoost...
 MAE (Erro médio absoluto): 3.68
 RMSE (Raiz do Erro Quadrático médio): 5.22
5. Calculando SHAP values...
6. Preparando dados finais para o PowerBI...
7. O processo foi concluido
